In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf

In [5]:
# Load model
model_lr = tf.keras.models.load_model('model/lr_best_model.h5')
model_ann = tf.keras.models.load_model('model/ann_best_model.h5')
model_lr_tuned = tf.keras.models.load_model('model/tuned_lr_model.h5')
model_ann_tuned = tf.keras.models.load_model('model/tuned_ann_model.h5')

In [6]:
gender = input("Gender (Male/Female): ")
age = int(input("Age: "))
smoke = input("Smoke (Yes/No/Unknown): ")
bmi = float(input("BMI: "))
HbA1c = float(input("HbA1c level: "))
glucose = float(input("Blood Glucose Level: "))

# Data preparation
entry = {
	'gender': gender,
	'age': age,
	'smoke': smoke,
	'bmi': bmi,
	'HbA1c_level': HbA1c,
	'blood_glucose_level': glucose
}
# Convert to DataFrame for further processing
input_df = pd.DataFrame(entry, index=[0])
input_df

,gender,age,smoke,bmi,HbA1c_level,blood_glucose_level
0,Male,28,Yes,26.0,6.0,180.0


Prepare

In [7]:
# Load original dataset
df = pd.read_csv('data/diabetes_prediction_dataset.csv')

In [8]:
# Pick min and max value
min_age = df['age'].min()
max_age = df['age'].max()
min_bmi = df['bmi'].min()
max_bmi = df['bmi'].max()
min_HbA1c = df['HbA1c_level'].min()
max_HbA1c = df['HbA1c_level'].max()
min_glucose = df['blood_glucose_level'].min()
max_glucose = df['blood_glucose_level'].max()

# Dataframe
minmax = pd.DataFrame({
    'Column': ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level'],
    'min': [min_age, min_bmi, min_HbA1c, min_glucose],
    'max': [max_age, max_bmi, max_HbA1c, max_glucose]
})
minmax

,Column,min,max
0,age,0.08,80.00
1,bmi,10.01,95.69
2,HbA1c_level,3.50,9.00
3,blood_glucose_level,80.00,300.00


Preprocessing

In [9]:
# gender
gender_mapping = {'Male': 1, 'Female': 0}
gender_mapped = gender_mapping[gender]

# smoke
def smoke_mapping(smoke):
  mapping = {
		'Yes': [1, 0, 0],
		'No': [0, 1, 0],
		'Unknown': [0, 0, 1]
	}
  return mapping.get(smoke, [0, 0, 0])
smoke_mapped = smoke_mapping(smoke)
smoke_mapped = np.array(smoke_mapped).astype("int32")
# Ensure smoke_mapped is a list of three elements
if len(smoke_mapped) != 3:
		raise ValueError("Invalid smoke input. Please enter 'Yes', 'No', or 'Unknown'.")

# age
age_normalized = (age - min_age) / (max_age - min_age)

# bmi
if bmi > 0 and bmi <= 18.5:
  bmi_binned = 'Underweight'
elif bmi > 18.5 and bmi <= 22.9:
  bmi_binned = 'Normal'
elif bmi > 22.9 and bmi <= 24.9:
  bmi_binned = 'Overweight'
elif bmi > 24.9 and bmi <= 29.9:
  bmi_binned = 'Obese Class I'
elif bmi > 29.9:
  bmi_binned = 'Obese Class II'
else:
  print("Input your correct BMI")

bmi_mapping = {'Underweight': 0, 'Normal': 1, 'Overweight': 2, 'Obese Class I': 3, 'Obese Class II': 4}
bmi_mapped = bmi_mapping[bmi_binned]

bmi_normalized = (bmi_mapped - 0) / (4 - 0)

# HbA1c
HbA1c_normalized = (HbA1c - min_HbA1c) / (max_HbA1c - min_HbA1c)

# glucose
glucose_transformed = np.log(df['blood_glucose_level'])
new_min_glucose = glucose_transformed.min()
new_max_glucose = glucose_transformed.max()

glucose_log_transformed = np.log(glucose)
glucose_normalized = (glucose_log_transformed - new_min_glucose) / (new_max_glucose - new_min_glucose)

In [10]:
print(new_max_glucose, new_min_glucose)

5.703782474656201 4.382026634673881


In [11]:
# Prepare data for prediction
data = pd.DataFrame({
  'gender': [gender_mapped],
  'age': [age_normalized],
  'bmi': [bmi_normalized],
  'HbA1c_level': [HbA1c_normalized],
  'blood_glucose_level': [glucose_normalized],
	'smoke_Yes': [smoke_mapped[0]],
  'smoke_No': [smoke_mapped[1]],
  'smoke_Unknown': [smoke_mapped[2]]  
})
data

,gender,age,bmi,HbA1c_level,blood_glucose_level,smoke_Yes,smoke_No,smoke_Unknown
0,1,0.349349,0.75,0.454545,0.613525,1,0,0


## Logistic Regression

### Without Tuning

In [12]:
# Make prediction using the preprocessed data
pred_lr = model_lr.predict(data)
pred_value_lr = pred_lr[0][0]

if pred_value_lr > 0.5:
	print("Predicted: Diabetes Positive")
else:
	print("Predicted: Diabetes Negative")

# Optionally, show the probability
print(f"Prediction probability: {pred_value_lr:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted: Diabetes Negative
Prediction probability: 0.3326


### With Tuning

In [13]:
# Make prediction using the preprocessed data
pred_lr_tuned = model_lr_tuned.predict(data)
pred_value_lr_tuned = pred_lr_tuned[0][0]

if pred_value_lr_tuned > 0.5:
	print("Predicted: Diabetes Positive")
else:
	print("Predicted: Diabetes Negative")

# Optionally, show the probability
print(f"Prediction probability: {pred_value_lr_tuned:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
Predicted: Diabetes Negative
Prediction probability: 0.3057


## Artificial Neural Network

### Without Tuning

In [14]:
# Make prediction using the ANN model
pred_ann = model_ann.predict(data)
pred_value_ann = pred_ann[0][0]
if pred_value_ann > 0.5:
	print("Predicted: Diabetes Positive")
else:
	print("Predicted: Diabetes Negative")
# Optionally, show the probability
print(f"ANN Prediction probability: {pred_value_ann:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Predicted: Diabetes Negative
ANN Prediction probability: 0.1172


### With Tuning

In [15]:
# Make prediction using the ANN model
pred_ann_tuned = model_ann_tuned.predict(data)
pred_value_ann_tuned = pred_ann_tuned[0][0]
if pred_value_ann_tuned > 0.5:
	print("Predicted: Diabetes Positive")
else:
	print("Predicted: Diabetes Negative")
# Optionally, show the probability
print(f"ANN Prediction probability: {pred_value_ann_tuned:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Predicted: Diabetes Negative
ANN Prediction probability: 0.0383


## Model Result Comparison

In [16]:
compare = pd.DataFrame({
		'Model': ['Logistic Regression', 'Tuned Logistic Regression', 'ANN', 'Tuned ANN'],
		'Prediction': [pred_value_lr, pred_value_lr_tuned, pred_value_ann, pred_value_ann_tuned]
})
compare

,Model,Prediction
0,Logistic Regression,0.332604
1,Tuned Logistic Regression,0.305715
2,ANN,0.117223
3,Tuned ANN,0.038274
